In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import glob

# ================= 配置区域 =================
# 1. 您的 .mat 数据文件夹路径
DATA_DIR = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal"

# 2. 您的 Excel 文件名 (请确保文件在当前目录下，或者写绝对路径)
LUT_PATH = "Freesurfer_LUT_alex_labels_jiayi.xlsx" 

# 通道定义
CH_UFA = 14       
CH_AMIDE = 225    
CH_MPRAGE = 341   

def load_colormap(excel_path):
    """直接从 Excel 加载 102 类配色"""
    try:
        # 修改点：使用 read_excel，并指定引擎 (以防万一)
        df = pd.read_excel(excel_path, engine='openpyxl')
        
        # 打印一下列名，确保读取正确 (调试用)
        # print("Excel Columns:", df.columns.tolist())
        
        # 寻找最大 Label ID 以建立颜色矩阵
        # 确保 label 列被当作数字处理，排除非数字干扰
        valid_labels = pd.to_numeric(df['one_hot_loc_alex_label'], errors='coerce').dropna()
        max_label = int(valid_labels.max()) + 10
        
        colors = np.zeros((max_label, 4)) # RGBA Matrix
        colors[0] = [0, 0, 0, 0] # 背景透明
        
        for _, row in df.iterrows():
            try:
                # 读取 Label ID
                lbl_val = row['one_hot_loc_alex_label']
                # 如果是字符串或无法转化的内容，跳过
                if pd.isna(lbl_val) or str(lbl_val).strip() == '[]':
                    continue
                    
                idx = int(lbl_val)
                
                # 读取 RGB 并归一化到 0-1
                r = row['R'] / 255.0
                g = row['G'] / 255.0
                b = row['B'] / 255.0
                
                colors[idx] = [r, g, b, 1.0] # Alpha = 1.0 (不透明)
            except Exception as e:
                # 某些行可能有格式问题，静默跳过即可
                continue
                
        return mcolors.ListedColormap(colors)
        
    except Exception as e:
        print(f"⚠️ 无法读取 Excel 文件 ({e})。")
        print("请检查：1. 文件路径是否正确 2. 是否安装了 openpyxl (pip install openpyxl)")
        return 'tab20'

# --- 主加载逻辑 ---

# 1. 查找数据
mat_files = sorted(list(Path(DATA_DIR).glob('*.mat')))
if not mat_files:
    print(f"⚠️ 警告: 在 {DATA_DIR} 未找到 .mat 文件。请检查路径。")
    # 为了防止报错停止，这里可以放一个伪数据逻辑，但真实运行请确保有文件
else:
    target_file = mat_files[0] 
    print(f"📂 正在加载数据: {target_file.name} ...")

    with h5py.File(target_file, 'r') as f:
        # 数据转置处理
        raw_data = f['data'][:]
        data_vol = np.moveaxis(raw_data, 0, -1)
        
        labels_vol = f['region_labels'][:]
        # 再次确认 Label 维度，如果 Label 也是 [C, X, Y, Z] 格式 (C=1)，也需要转置
        if labels_vol.ndim == 4 and labels_vol.shape[0] == 1:
            labels_vol = labels_vol[0, ...] # 降维成 [X, Y, Z]
        elif labels_vol.ndim == 4:
            labels_vol = np.moveaxis(labels_vol, 0, -1)
            
        print(f"✅ 数据加载完成。Data Shape: {data_vol.shape}")
        
    # 2. 加载 Excel 配色
    custom_cmap = load_colormap(LUT_PATH)
    print("✅ 配色表已就绪。")

In [ ]:
# ================= 绘图参数 =================
AXIS = 'Axial'       # 可选 'Axial', 'Coronal', 'Sagittal'
START_SLICE = 130    # 起始层
END_SLICE = 150      # 结束层
STEP = 2             # 每隔几层画一张

def normalize(img, p_min=1, p_max=99):
    """鲁棒归一化"""
    vmin, vmax = np.percentile(img, [p_min, p_max])
    return np.clip(img, vmin, vmax)

def get_slice(data, labels, axis, idx):
    """提取特定方向的切片并旋转至解剖学正位"""
    # 假设数据格式 (X, Y, Z, C) -> (384, 336, 256, 351)
    if axis == 'Axial': # 取 Z 轴 (Index 2)
        # 通常需要逆时针旋转 90 度以符合看片习惯
        sl_data = np.rot90(data[:, :, idx, :])
        sl_label = np.rot90(labels[:, :, idx]) if labels.ndim == 3 else np.rot90(labels[0, :, :, idx]) 
    elif axis == 'Coronal': # 取 Y 轴 (Index 1)
        sl_data = np.rot90(data[:, idx, :, :])
        sl_label = np.rot90(labels[:, idx, :])
    else: # Sagittal (Index 0)
        sl_data = np.rot90(data[idx, :, :, :])
        sl_label = np.rot90(labels[idx, :, :])
    return sl_data, sl_label

# 循环生成图表
print(f"正在生成 {AXIS} 视图 (层 {START_SLICE} - {END_SLICE})...")

for slice_idx in range(START_SLICE, END_SLICE, STEP):
    
    # 准备切片数据
    try:
        sl_data, sl_label = get_slice(data_vol, labels_vol, AXIS, slice_idx)
    except IndexError:
        print(f"层 {slice_idx} 超出范围，跳过。")
        continue

    # 提取各模态
    img_mprage = sl_data[..., CH_MPRAGE]
    img_ufa    = sl_data[..., CH_UFA]
    img_amide  = sl_data[..., CH_AMIDE]
    
    # 创建画布
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=150)
    fig.suptitle(f"Figure 2.1 Candidate: {AXIS} Slice {slice_idx}", fontsize=16, y=0.95)
    plt.subplots_adjust(wspace=0.1, hspace=0.15)
    
    # --- Panel A: Structural (MPRAGE) ---
    ax = axes[0, 0]
    ax.imshow(img_mprage, cmap='gray', aspect='equal')
    ax.set_title(r"$\bf{(A)}$ High-Res MPRAGE (0.65mm)", loc='left', fontsize=12)
    ax.axis('off')
    
    # --- Panel B: Segmentation Overlay ---
    ax = axes[0, 1]
    ax.imshow(img_mprage, cmap='gray', aspect='equal') # 底图
    masked_lbl = np.ma.masked_where(sl_label == 0, sl_label) # 隐藏背景
    ax.imshow(masked_lbl, cmap=custom_cmap, alpha=0.6, interpolation='nearest')
    ax.set_title(r"$\bf{(B)}$ 102-Class Segmentation Overlay", loc='left', fontsize=12)
    ax.axis('off')
    
    # --- Panel C: Microstructure (uFA) ---
    ax = axes[1, 0]
    im_c = ax.imshow(img_ufa, cmap='magma', aspect='equal')
    ax.set_title(r"$\bf{(C)}$ QTI Microstructure ($\mu$FA)", loc='left', fontsize=12)
    ax.axis('off')
    cbar = plt.colorbar(im_c, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("$\mu$FA (a.u.)", rotation=270, labelpad=15)
    
    # --- Panel D: Biochemistry (CEST) ---
    ax = axes[1, 1]
    # 关键：interpolation='nearest' 显式展示低分辨率锯齿
    im_d = ax.imshow(img_amide, cmap='inferno', aspect='equal', interpolation='nearest')
    ax.set_title(r"$\bf{(D)}$ CEST Amide (Low-Res)", loc='left', fontsize=12)
    ax.axis('off')
    cbar = plt.colorbar(im_d, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Amide Contrast (a.u.)", rotation=270, labelpad=15)
    
    plt.show() # 直接显示，方便挑选